# Data augmentation notebook

Notebook này dùng để **lấy dữ liệu bài báo thật từ web và xử lý dữ liệu trong RAM**. Trong lúc chạy, dữ liệu được giữ bằng list/dict Python để dễ debug, lọc, xem preview. Chỉ ở cell cuối mới ghi `articles_raw.jsonl`, `articles_clean.jsonl`, log download và report ra `runs/notebooks/<run_id>/`.

Notebook này không gọi teacher LLM, student LLM, embedding, RAG hoặc DB. Sau khi có `articles_clean.jsonl`, các milestone tiếp theo chạy bằng Python/CLI.

## Luồng chạy

1. Cấu hình các trang nguồn tài chính Việt Nam.
2. Truy cập trang nguồn thật và discovery URL bài báo.
3. Download HTML bài báo thật vào RAM.
4. Parse HTML, chuẩn hóa text, bóc metadata ticker/event keyword, lọc bài lỗi/ngắn/trùng.
5. Preview dữ liệu sạch.
6. Ghi JSONL cuối cùng để các bước sau dùng.

In [1]:
from __future__ import annotations

import json
from pathlib import Path

from finevent.ingestion.discovery import (
    DEFAULT_DISCOVERY_KEYWORD_HINTS,
    DEFAULT_SEED_PAGES,
    SeedPage,
    discover_url_candidates,
)
from finevent.ingestion.download import UrlCandidate, build_research_session, fetch_url_candidates
from finevent.ingestion.metadata import (
    DEFAULT_EVENT_KEYWORD_TAXONOMY_PATH,
    extract_event_keyword_matches,
    extract_event_keywords,
    extract_event_subtype_hints,
    extract_event_type_hints,
    extract_sector_hints,
    extract_tickers_and_companies,
    load_company_dictionary,
    load_event_keyword_taxonomy,
)
from finevent.ingestion.models import CleanArticleRecord, RawArticleRecord
from finevent.ingestion.parsers import parse_article_html
from finevent.ingestion.reporting import build_data_quality_summary, write_data_quality_summary
from finevent.ingestion.text import canonical_url, normalize_text, stable_article_id, text_hash
from finevent.jsonl import write_jsonl
from finevent.logging_utils import create_run_id, utc_now_iso

RUN_ID = create_run_id("data_aug")
RUN_ROOT = Path("runs/notebooks") / RUN_ID
OUTPUT_RAW_PATH = RUN_ROOT / "data/raw/articles_raw.jsonl"
OUTPUT_CLEAN_PATH = RUN_ROOT / "data/processed/articles_clean.jsonl"
OUTPUT_DISCOVERED_PATH = RUN_ROOT / "data/raw/discovered_urls.jsonl"
OUTPUT_DOWNLOAD_LOG_PATH = RUN_ROOT / "data/raw/download_log.jsonl"
OUTPUT_REPORT_PATH = RUN_ROOT / "reports/data_quality_summary.md"
OUTPUT_SAMPLE_PATH = RUN_ROOT / "data/processed/articles_clean_sample.jsonl"
OUTPUT_HTML_DIR = RUN_ROOT / "data/raw/html"

REAL_DOWNLOAD = True
SAVE_HTML_SNAPSHOTS = True
REQUEST_TIMEOUT_SECONDS = 25.0
MAX_DISCOVERED_URLS = 80
MAX_ARTICLES_TO_DOWNLOAD = 25
MIN_TEXT_CHARS = 300
SAMPLE_LIMIT = 10

SEED_PAGES = DEFAULT_SEED_PAGES

# N?u discovery t? trang chuy?n m?c b? website ch?n, th?m URL b?i b?o th?t v?o ??y.
# Notebook v?n s? download v? x? l? nh? d? li?u th?t, kh?ng d?ng fixture local.
MANUAL_ARTICLE_URLS = [
    # {"source": "cafef", "url": "https://cafef.vn/..."},
]

KEYWORD_HINTS = list(DEFAULT_DISCOVERY_KEYWORD_HINTS)

for path_item in [OUTPUT_RAW_PATH.parent, OUTPUT_CLEAN_PATH.parent, OUTPUT_REPORT_PATH.parent, OUTPUT_HTML_DIR]:
    path_item.mkdir(parents=True, exist_ok=True)

session = build_research_session()

print(json.dumps({
    "run_id": RUN_ID,
    "run_root": str(RUN_ROOT),
    "real_download": REAL_DOWNLOAD,
    "save_html_snapshots": SAVE_HTML_SNAPSHOTS,
    "max_discovered_urls": MAX_DISCOVERED_URLS,
    "max_articles_to_download": MAX_ARTICLES_TO_DOWNLOAD,
}, ensure_ascii=False, indent=2))

{
  "run_id": "data_aug_20260621T165131Z_0d2bf624",
  "run_root": "runs\\notebooks\\data_aug_20260621T165131Z_0d2bf624"
}


## 1. Discovery URL bài báo thật

Cell này mở các trang chuyên mục thật, trích link bài báo, chấm điểm link theo từ khóa tài chính/doanh nghiệp và giữ các URL có khả năng chứa sự kiện doanh nghiệp.

In [2]:
if not REAL_DOWNLOAD:
    raise RuntimeError("REAL_DOWNLOAD ?ang False. Notebook n?y ???c c?u h?nh ?? ki?m th? lu?ng l?y d? li?u th?t.")

seed_pages = [SeedPage.from_dict(record) for record in SEED_PAGES]
manual_candidates = [
    UrlCandidate(
        source=str(record["source"]),
        url=str(record["url"]),
        link_text=str(record.get("link_text") or "manual candidate"),
        score=int(record.get("score") or 999),
        seed_url="manual",
        discovered_at=utc_now_iso(),
    )
    for record in MANUAL_ARTICLE_URLS
]

discovery_result = discover_url_candidates(
    seed_pages=seed_pages,
    manual_candidates=manual_candidates,
    max_candidates=MAX_DISCOVERED_URLS,
    timeout_seconds=REQUEST_TIMEOUT_SECONDS,
    keyword_hints=KEYWORD_HINTS,
    session=session,
)
discovered_candidates = discovery_result.candidates
discovery_diagnostics = discovery_result.diagnostics

if not discovered_candidates:
    raise RuntimeError("Kh?ng t?m ???c URL b?i b?o th?t n?o. Ki?m tra m?ng/source ho?c th?m MANUAL_ARTICLE_URLS.")

print(json.dumps({
    "candidate_count": len(discovered_candidates),
    "diagnostics": [item.to_dict() for item in discovery_diagnostics],
    "top_candidates": [candidate.to_dict() for candidate in discovered_candidates[:10]],
}, ensure_ascii=False, indent=2))

{
  "seed_count": 4,
  "diagnostics": [
    {
      "source": "cafef",
      "seed_url": "https://cafef.vn/thi-truong-chung-khoan.chn",
      "status_code": 200,
      "error": null,
      "candidate_count": 74
    },
    {
      "source": "vietstock",
      "seed_url": "https://vietstock.vn/chung-khoan.htm",
      "status_code": 200,
      "error": null,
      "candidate_count": 414
    },
    {
      "source": "tinnhanhchungkhoan",
      "seed_url": "https://www.tinnhanhchungkhoan.vn/chung-khoan/",
      "status_code": 200,
      "error": null,
      "candidate_count": 166
    },
    {
      "source": "nhadautu",
      "seed_url": "https://nhadautu.vn/doanh-nghiep-d3.html",
      "status_code": 200,
      "error": null,
      "candidate_count": 92
    }
  ],
  "candidate_count": 80,
  "top_candidates": [
    {
      "source": "cafef",
      "url": "https://cafef.vn/co-dong-lien-quan-lanh-dao-vinasun-khong-ban-het-luong-co-phieu-vns-da-dang-ky-188260621162217363.chn",
      "link_text

## 2. Download HTML bài báo vào RAM

Cell này download từng URL đã discovery. HTML được giữ trong `downloaded_pages` để xử lý tiếp trong RAM. Nếu bật `SAVE_HTML_SNAPSHOTS`, HTML chỉ được lưu ra file ở bước xuất artifact.

In [3]:
fetched_pages = fetch_url_candidates(
    discovered_candidates,
    timeout_seconds=REQUEST_TIMEOUT_SECONDS,
    max_records=MAX_ARTICLES_TO_DOWNLOAD,
    session=session,
)

downloaded_pages = []
download_log = []
for index, page in enumerate(fetched_pages, start=1):
    record = page.to_log_dict()
    record["rank"] = index
    download_log.append(record)
    if page.error is None and page.html:
        downloaded_pages.append({"candidate": page.candidate.to_dict(), "html": page.html})

success_count = sum(1 for record in download_log if record["error"] is None)
if success_count == 0:
    raise RuntimeError("Kh?ng download th?nh c?ng b?i b?o th?t n?o. Ki?m tra m?ng/source ho?c th?m MANUAL_ARTICLE_URLS.")

print(json.dumps({
    "attempt_count": len(download_log),
    "success_count": success_count,
    "error_count": len(download_log) - success_count,
    "successful_urls": [record["url"] for record in download_log if record["error"] is None][:10],
}, ensure_ascii=False, indent=2))

{
  "attempt_count": 25,
  "success_count": 25,
  "error_count": 0,
  "successful_urls": [
    "https://cafef.vn/co-dong-lien-quan-lanh-dao-vinasun-khong-ban-het-luong-co-phieu-vns-da-dang-ky-188260621162217363.chn",
    "https://cafef.vn/chung-khoan-tuan-qua-tam-diem-co-phieu-ho-vin-nvl-pc1-188260620153727182.chn",
    "https://cafef.vn/xu-phat-hai-cong-ty-chung-khoan-188260620124716619.chn",
    "https://cafef.vn/mot-cong-ty-chung-khoan-muon-thanh-ly-co-phieu-tren-481-tai-khoan-khach-hang-ubcknn-lap-tuc-tuyt-coi-188260620091321236.chn",
    "https://www.tinnhanhchungkhoan.vn/nhin-lai-dien-bien-nhom-co-phieu-duoc-cac-cong-ty-chung-khoan-khuyen-nghi-tuan-qua-post392690.html",
    "https://www.tinnhanhchungkhoan.vn/dau-tu-ha-tang-ky-thuat-tphcm-cii-thay-doi-phuong-an-su-dung-von-thu-duoc-tu-dot-chao-ban-trai-phieu-chuyen-doi-post392506.html",
    "https://www.tinnhanhchungkhoan.vn/cong-ty-co-phan-tap-doan-dau-khi-an-pha-asp-hose-post387526.html",
    "https://www.tinnhanhchungkhoan.vn/c

## 3. Parse, chuẩn hóa và bóc metadata trong RAM

Cell này biến HTML thành raw records và clean records. Các bước lọc gồm: bài quá ngắn, parse lỗi, nội dung trùng hash. Metadata được bóc bằng dictionary ticker và taxonomy keyword hiện có trong project.

In [4]:
company_entries = load_company_dictionary("data/dictionaries/ticker_company_map.csv")
taxonomy_entries = load_event_keyword_taxonomy(DEFAULT_EVENT_KEYWORD_TAXONOMY_PATH)

raw_records = []
clean_records = []
seen_hashes = set()
duplicate_count = 0

for page in downloaded_pages:
    candidate = page["candidate"]
    source = candidate["source"]
    url = candidate["url"]
    html = page["html"]
    parsed = parse_article_html(html, source=source, url=url)
    normalized_text = normalize_text(parsed.body_text)
    article_id = stable_article_id(source, url, normalized_text or parsed.title or url)
    parse_warnings = list(parsed.warnings)
    parse_status = "success"
    if len(normalized_text) < MIN_TEXT_CHARS:
        parse_status = "too_short"
        parse_warnings.append("text_below_min_chars")

    raw_record = RawArticleRecord(
        article_id=article_id,
        source=source,
        url=canonical_url(url),
        title=parsed.title,
        published_at=parsed.published_at,
        author=parsed.author,
        http_status=200,
        crawl_time=utc_now_iso(),
        html_path=None,
        raw_text=normalized_text,
        parse_status=parse_status,
        parse_warnings=parse_warnings,
    )
    raw_records.append(raw_record)

    if parse_status != "success":
        continue
    content_hash = text_hash(normalized_text)
    if content_hash in seen_hashes:
        duplicate_count += 1
        continue
    seen_hashes.add(content_hash)

    metadata_text = "\n".join(part for part in [parsed.title, normalized_text] if part)
    tickers, company_names = extract_tickers_and_companies(metadata_text, company_entries)
    keyword_matches = extract_event_keyword_matches(metadata_text, taxonomy_entries)
    matched_keywords = extract_event_keywords(metadata_text, taxonomy_entries=taxonomy_entries)
    clean_records.append(
        CleanArticleRecord(
            article_id=article_id,
            source=source,
            url=canonical_url(url),
            raw_html_path="",
            title=parsed.title,
            published_at=parsed.published_at,
            text=normalized_text,
            tickers_hint=tickers,
            company_names_hint=company_names,
            sector_hints=extract_sector_hints(tickers, company_entries),
            event_keywords=matched_keywords,
            event_type_hints=extract_event_type_hints(keyword_matches),
            event_subtype_hints=extract_event_subtype_hints(keyword_matches),
            language="vi",
            content_hash=content_hash,
            text_char_count=len(normalized_text),
        )
    )

if not clean_records:
    raise RuntimeError("Download có kết quả nhưng không tạo được clean record nào. Xem raw_records để debug parser/filter.")

quality_summary = {
    "downloaded_pages": len(downloaded_pages),
    "raw_count": len(raw_records),
    "clean_count": len(clean_records),
    "duplicate_count": duplicate_count,
    "too_short_count": sum(1 for record in raw_records if record.parse_status == "too_short"),
    "with_ticker_count": sum(1 for record in clean_records if record.tickers_hint),
    "with_event_hint_count": sum(1 for record in clean_records if record.event_type_hints),
}
print(json.dumps(quality_summary, ensure_ascii=False, indent=2))

{
  "downloaded_pages": 25,
  "raw_count": 25,
  "clean_count": 25,
  "duplicate_count": 0,
  "too_short_count": 0,
  "with_ticker_count": 18,
  "with_event_hint_count": 23
}


## 4. Preview dữ liệu sạch trước khi ghi file

Kiểm tra nhanh title, ticker, event type/subtype và độ dài text ngay trong notebook. Nếu dữ liệu chưa ổn, chỉnh seed/source/filter ở các cell trên rồi chạy lại.

In [5]:
preview = []
for record in clean_records[:SAMPLE_LIMIT]:
    payload = record.to_dict()
    preview.append({
        "article_id": payload["article_id"],
        "source": payload["source"],
        "title": payload["title"],
        "tickers_hint": payload["tickers_hint"],
        "company_names_hint": payload["company_names_hint"],
        "event_type_hints": payload["event_type_hints"],
        "event_subtype_hints": payload["event_subtype_hints"],
        "text_char_count": payload["text_char_count"],
        "url": payload["url"],
    })
print(json.dumps(preview, ensure_ascii=False, indent=2))

[
  {
    "article_id": "cafef_c15072521157",
    "source": "cafef",
    "title": "Cổ đông liên quan lãnh đạo Vinasun không bán hết lượng cổ phiếu VNS đã đăng ký",
    "tickers_hint": [],
    "company_names_hint": [],
    "event_type_hints": [
      "DIVIDEND_SHAREHOLDER"
    ],
    "event_subtype_hints": [
      "MAJOR_SHAREHOLDER_TRANSACTION"
    ],
    "text_char_count": 2252,
    "url": "https://cafef.vn/co-dong-lien-quan-lanh-dao-vinasun-khong-ban-het-luong-co-phieu-vns-da-dang-ky-188260621162217363.chn"
  },
  {
    "article_id": "cafef_f9a04165d135",
    "source": "cafef",
    "title": "Chứng khoán tuần qua: Tâm điểm cổ phiếu “họ” Vin, NVL, PC1",
    "tickers_hint": [
      "BSR",
      "CTG",
      "DCM",
      "FPT",
      "HCM",
      "HDB",
      "MBB",
      "MWG",
      "NVL",
      "PC1",
      "STB",
      "TCB",
      "VCB",
      "VHM",
      "VIC",
      "VPB",
      "VRE"
    ],
    "company_names_hint": [
      "Binh Son Refining and Petrochemical",
      "FPT Corpo

## 5. Ghi JSONL cuối cùng

Chỉ cell này mới ghi dữ liệu ra file. Các file trong `runs/notebooks/<run_id>/` là artifact sandbox để bạn kiểm tra. Khi ổn mới promote `articles_clean.jsonl` sang `data/processed/articles_clean.jsonl`.

In [6]:
write_jsonl(OUTPUT_DISCOVERED_PATH, (candidate.to_dict() for candidate in discovered_candidates))
write_jsonl(OUTPUT_DOWNLOAD_LOG_PATH, download_log)
write_jsonl(OUTPUT_RAW_PATH, (record.to_dict() for record in raw_records))
write_jsonl(OUTPUT_CLEAN_PATH, (record.to_dict() for record in clean_records))
write_jsonl(OUTPUT_SAMPLE_PATH, (record.to_dict() for record in clean_records[:SAMPLE_LIMIT]))

if SAVE_HTML_SNAPSHOTS:
    for index, page in enumerate(downloaded_pages, start=1):
        source = page["candidate"]["source"]
        html_path = OUTPUT_HTML_DIR / f"{index:03d}_{source}.html"
        html_path.write_text(page["html"], encoding="utf-8")

report_summary = build_data_quality_summary(
    raw_records=raw_records,
    clean_records=clean_records,
    duplicate_count=duplicate_count,
)
write_data_quality_summary(OUTPUT_REPORT_PATH, report_summary)

print(json.dumps({
    "run_root": str(RUN_ROOT),
    "discovered_urls_path": str(OUTPUT_DISCOVERED_PATH),
    "download_log_path": str(OUTPUT_DOWNLOAD_LOG_PATH),
    "raw_path": str(OUTPUT_RAW_PATH),
    "clean_path": str(OUTPUT_CLEAN_PATH),
    "sample_path": str(OUTPUT_SAMPLE_PATH),
    "report_path": str(OUTPUT_REPORT_PATH),
    "clean_count": len(clean_records),
}, ensure_ascii=False, indent=2))

{
  "run_root": "runs\\notebooks\\data_aug_20260621T165131Z_0d2bf624",
  "discovered_urls_path": "runs\\notebooks\\data_aug_20260621T165131Z_0d2bf624\\data\\raw\\discovered_urls.jsonl",
  "download_log_path": "runs\\notebooks\\data_aug_20260621T165131Z_0d2bf624\\data\\raw\\download_log.jsonl",
  "raw_path": "runs\\notebooks\\data_aug_20260621T165131Z_0d2bf624\\data\\raw\\articles_raw.jsonl",
  "clean_path": "runs\\notebooks\\data_aug_20260621T165131Z_0d2bf624\\data\\processed\\articles_clean.jsonl",
  "sample_path": "runs\\notebooks\\data_aug_20260621T165131Z_0d2bf624\\data\\processed\\articles_clean_sample.jsonl",
  "report_path": "runs\\notebooks\\data_aug_20260621T165131Z_0d2bf624\\reports\\data_quality_summary.md",
  "clean_count": 25
}


## Sau khi có JSONL sạch

Khi `runs/notebooks/<run_id>/data/processed/articles_clean.jsonl` ổn, copy/promote file này sang `data/processed/articles_clean.jsonl`. Sau đó chạy các milestone tiếp theo bằng Python/CLI, ví dụ:

```powershell
python -m finevent.labeling generate-prompts --articles-path data/processed/articles_clean.jsonl
python -m finevent.labeling run-teacher --prompt-path data/labels/teacher_prompts.jsonl
python -m finevent.labeling validate --articles-path data/processed/articles_clean.jsonl
python -m finevent.rag prepare --articles-path data/processed/articles_clean.jsonl --embedding-provider openai_compatible
```

Notebook này cố ý không truy cập DB và không chạy các bước model để phần lấy dữ liệu thật dễ debug hơn.